# OOD Evaluation: NAICS-2 Cross-Encoder

In [ ]:
!pip install -q transformers sentencepiece scikit-learn pandas tqdm

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
MODEL_NAME      = "microsoft/deberta-v3-small"
MODEL_PATH      = "best_model.pt"
OOD_PATH        = "ood_dataset_official.csv"
TAXONOMY_PATH   = "ExioNAICS.csv"
RESULTS_DIR     = "results_ood"
MAX_LENGTH      = 192
QUERY_BATCH     = 16
MAX_EXAMPLES    = 25
SEED            = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model:        {MODEL_NAME}  (weights: {MODEL_PATH})")
print(f"OOD dataset:  {OOD_PATH}")
print(f"Taxonomy:     {TAXONOMY_PATH}")
print(f"Max length:   {MAX_LENGTH}")
print(f"Query batch:  {QUERY_BATCH}  (= {QUERY_BATCH * 20} sequences/forward)")

In [ ]:
df_raw = pd.read_csv(TAXONOMY_PATH)

SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
}

naics2_raw = df_raw[['NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description']].drop_duplicates(subset='NAICS_2 Code').copy()
naics2_raw['NAICS_2 Code'] = naics2_raw['NAICS_2 Code'].astype(str)
naics2_raw['sector_code'] = naics2_raw['NAICS_2 Code'].map(SECTOR_MERGE).fillna(naics2_raw['NAICS_2 Code'])

naics2_corpus = naics2_raw.drop_duplicates(subset='sector_code').copy()
naics2_corpus = naics2_corpus.sort_values('sector_code').reset_index(drop=True)

sector_codes  = naics2_corpus['sector_code'].tolist()
sector_titles = naics2_corpus['NAICS_2 Title'].tolist()
code_to_idx   = {code: i for i, code in enumerate(sector_codes)}
NUM_CLASSES   = len(sector_codes)

naics6_titles = df_raw[['NAICS Code', 'NAICS Title']].drop_duplicates(subset='NAICS Code').dropna()
naics6_titles['NAICS Code'] = naics6_titles['NAICS Code'].astype(str)
naics6_titles['sector'] = naics6_titles['NAICS Code'].str[:2].map(SECTOR_MERGE).fillna(naics6_titles['NAICS Code'].str[:2])

sector_to_examples = {}
for sector, grp in naics6_titles.groupby('sector'):
    titles = [t.strip() for t in grp['NAICS Title'].astype(str).tolist() if t.strip()]
    sector_to_examples[sector] = titles[:MAX_EXAMPLES]

corpus_texts = []
for i, code in enumerate(sector_codes):
    title = sector_titles[i]
    description = naics2_corpus['NAICS_2 Description'].iloc[i]
    description = str(description) if pd.notna(description) else ""
    examples = sector_to_examples.get(code, [])
    if examples:
        text = f"{title}. Examples: {'; '.join(examples)}. {description}"
    else:
        text = f"{title}. {description}"
    corpus_texts.append(text)

print(f"NAICS-2 sectors: {NUM_CLASSES}")
for i, (code, title) in enumerate(zip(sector_codes, sector_titles)):
    print(f"  {i:>2}: {code:>5} -> {title}")

print(f"\nEnriched corpus length (chars): "
      f"min={min(len(t) for t in corpus_texts)}, "
      f"max={max(len(t) for t in corpus_texts)}, "
      f"mean={int(np.mean([len(t) for t in corpus_texts]))}")

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df_ood = pd.read_csv(OOD_PATH)
df_ood['naics2_code'] = df_ood['naics2_code'].astype(str)

before = len(df_ood)
df_ood = df_ood[df_ood['naics2_code'].isin(code_to_idx)].reset_index(drop=True)
dropped_unknown = before - len(df_ood)
if dropped_unknown:
    print(f"Dropped {dropped_unknown} OOD rows with sector codes not in trained label space")

df_ood['naics2_idx'] = df_ood['naics2_code'].map(code_to_idx).astype(int)
df_ood['query_text'] = df_ood['model_input'].apply(preprocess_text)

df_ood = df_ood[df_ood['query_text'].str.len() > 0].reset_index(drop=True)

queries = df_ood['query_text'].tolist()
labels  = df_ood['naics2_idx'].tolist()

print(f"OOD test samples: {len(df_ood)}")
print(f"\nClass distribution (OOD):")
for code in sector_codes:
    cnt = (df_ood['naics2_code'] == code).sum()
    print(f"  {code:>5}  {sector_titles[code_to_idx[code]]:<60}  {cnt}")

print(f"\nQuery length stats (chars):")
print(df_ood['query_text'].str.len().describe().round(0).to_string())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, torch_dtype=torch.float32
).to(device)

state = torch.load(MODEL_PATH, map_location=device)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
elif isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']

missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Loaded weights from {MODEL_PATH}")
print(f"  Missing keys:    {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")
if missing:
    print(f"  First missing:   {missing[:5]}")
if unexpected:
    print(f"  First unexpected:{unexpected[:5]}")

model.eval()
print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
@torch.no_grad()
def evaluate_ood(model, tokenizer, queries, labels, corpus_texts,
                 max_length=192, query_batch=16):
    model.eval()
    n = len(queries)
    num_classes = len(corpus_texts)
    use_amp = (device.type == "cuda")

    all_top5 = np.zeros((n, 5), dtype=np.int64)
    all_logits = np.zeros((n, num_classes), dtype=np.float32)

    for start in tqdm(range(0, n, query_batch), desc="Scoring OOD"):
        batch_queries = queries[start:start + query_batch]
        bs = len(batch_queries)

        pair_q = [q for q in batch_queries for _ in range(num_classes)]
        pair_d = corpus_texts * bs

        enc = tokenizer(
            pair_q, pair_d,
            max_length=max_length, truncation=True,
            padding=True, return_tensors='pt'
        )
        enc = {k: v.to(device, non_blocking=True)
               for k, v in enc.items() if k in ['input_ids', 'attention_mask']}

        if use_amp:
            with torch.cuda.amp.autocast(dtype=torch.float16):
                logits = model(**enc).logits.squeeze(-1)
        else:
            logits = model(**enc).logits.squeeze(-1)

        logits = logits.float().reshape(bs, num_classes)
        topk = logits.topk(5, dim=1).indices.cpu().numpy()

        all_top5[start:start + bs] = topk
        all_logits[start:start + bs] = logits.cpu().numpy()

    preds = all_top5[:, 0].tolist()
    labels_arr = np.asarray(labels)

    top1 = float(np.mean(all_top5[:, 0] == labels_arr))
    top3 = float(np.mean([labels_arr[i] in all_top5[i, :3] for i in range(n)]))
    top5 = float(np.mean([labels_arr[i] in all_top5[i, :5] for i in range(n)]))
    macro_f1    = f1_score(labels, preds, average='macro',    zero_division=0)
    weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)

    return {
        'top1': top1, 'top3': top3, 'top5': top5,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'preds': preds, 'top5_indices': all_top5, 'logits': all_logits,
    }


print("Evaluation function ready.")

In [ ]:
results = evaluate_ood(
    model, tokenizer, queries, labels, corpus_texts,
    max_length=MAX_LENGTH, query_batch=QUERY_BATCH,
)

print("\n" + "=" * 70)
print(f"OOD evaluation: {len(queries)} samples, {NUM_CLASSES} classes")
print("=" * 70)
print(f"  Top-1 accuracy: {results['top1']:.4f}")
print(f"  Top-3 accuracy: {results['top3']:.4f}")
print(f"  Top-5 accuracy: {results['top5']:.4f}")
print(f"  Macro F1:       {results['macro_f1']:.4f}")
print(f"  Weighted F1:    {results['weighted_f1']:.4f}")
print("=" * 70)

In [ ]:
target_names = [f"{sector_codes[i]} ({sector_titles[i][:40]})" for i in range(NUM_CLASSES)]
present_idx = sorted(set(labels))

report_str = classification_report(
    labels, results['preds'],
    labels=present_idx,
    target_names=[target_names[i] for i in present_idx],
    digits=4, zero_division=0,
)
print("Per-class classification report:\n")
print(report_str)

report_dict = classification_report(
    labels, results['preds'],
    labels=present_idx,
    target_names=[target_names[i] for i in present_idx],
    digits=4, zero_division=0, output_dict=True,
)

In [ ]:
predictions_df = df_ood[['company name', 'description', 'naics2_code', 'naics2_title']].copy()
predictions_df['pred_idx']    = results['preds']
predictions_df['pred_code']   = predictions_df['pred_idx'].map(lambda i: sector_codes[i])
predictions_df['pred_title']  = predictions_df['pred_idx'].map(lambda i: sector_titles[i])
predictions_df['correct_top1'] = predictions_df['naics2_code'] == predictions_df['pred_code']

for k in range(5):
    predictions_df[f'top{k+1}_code'] = [sector_codes[results['top5_indices'][i, k]] for i in range(len(df_ood))]

predictions_path = os.path.join(RESULTS_DIR, 'ood_predictions.csv')
predictions_df.to_csv(predictions_path, index=False)
print(f"Per-row predictions saved to: {predictions_path}")

summary = {
    'n_samples':         len(queries),
    'n_classes_in_test': len(present_idx),
    'n_classes_total':   NUM_CLASSES,
    'top1':              results['top1'],
    'top3':              results['top3'],
    'top5':              results['top5'],
    'macro_f1':          results['macro_f1'],
    'weighted_f1':       results['weighted_f1'],
    'per_class':         report_dict,
}
summary_path = os.path.join(RESULTS_DIR, 'ood_metrics.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary metrics saved to:    {summary_path}")

with open(os.path.join(RESULTS_DIR, 'classification_report.txt'), 'w') as f:
    f.write(f"OOD evaluation on {len(queries)} samples\n")
    f.write(f"Top-1 {results['top1']:.4f} | Top-3 {results['top3']:.4f} | Top-5 {results['top5']:.4f} | "
            f"Macro F1 {results['macro_f1']:.4f} | Weighted F1 {results['weighted_f1']:.4f}\n\n")
    f.write(report_str)
print(f"Classification report saved to: {os.path.join(RESULTS_DIR, 'classification_report.txt')}")

In [ ]:
cm = confusion_matrix(labels, results['preds'], labels=list(range(NUM_CLASSES)))

print("Confusion matrix (rows=true, cols=pred), sectors with at least 1 OOD sample:")
header = "true \\ pred  | " + " ".join(f"{c:>5}" for c in sector_codes) + "  |  total"
print(header)
print("-" * len(header))
for i in range(NUM_CLASSES):
    row_total = cm[i].sum()
    if row_total == 0:
        continue
    row = " ".join(f"{cm[i, j]:>5}" for j in range(NUM_CLASSES))
    print(f"{sector_codes[i]:>11}  | {row}  |  {row_total}")

print("\nMost common confusions (true -> pred, at least 30 cases):")
confusions = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] >= 30:
            confusions.append((cm[i, j], sector_codes[i], sector_titles[i],
                               sector_codes[j], sector_titles[j]))
confusions.sort(reverse=True)
for cnt, tc, tt, pc, pt in confusions[:20]:
    print(f"  {cnt:>5}  {tc:>5} {tt[:35]:<35} -> {pc:>5} {pt[:35]}")

In [ ]:
import shutil

zip_base = "ood_results"
zip_path = shutil.make_archive(zip_base, 'zip', RESULTS_DIR)
print(f"Zipped {RESULTS_DIR}/ -> {zip_path}")
print(f"Contents:")
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<35}  {size_kb:>10.1f} KB")

try:
    from google.colab import files
    files.download(zip_path)
    print(f"\nDownload triggered for: {zip_path}")
except ImportError:
    print(f"\nNot running in Colab - results are at: {os.path.abspath(zip_path)}")